In [13]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# =============================
# 1. Install & Imports
# =============================
!pip install pytorch-tabnet -q

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

from pytorch_tabnet.tab_model import TabNetClassifier
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Load the dataset
file_path = '/content/drive/MyDrive/Crime Prediction Datasets/Final crime stats dataset.csv'  # Update with your actual file path
df = pd.read_csv(file_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Torch version: 2.9.0+cu126
CUDA available: True


In [17]:
# =============================
# 2. Preprocess & train/valid split
# =============================

# 🔁 Choose which label you want to predict
target_col = "Crime_Level"        # <-- change to "Recovery_Level" if you want that instead

# Drop the target and the *other* level column to avoid leakage
drop_cols = [target_col]
if target_col == "Crime_Level" and "Recovery_Level" in df.columns:
    drop_cols.append("Recovery_Level")
elif target_col == "Recovery_Level" and "Crime_Level" in df.columns:
    drop_cols.append("Crime_Level")

X = df.drop(columns=drop_cols)
y = df[target_col]

# Treat "Unit Name" as categorical, everything else numeric
cat_cols = ["Unit Name"]
for c in cat_cols:
    X[c] = X[c].astype("category")

# One-hot encode Unit Name
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Just in case: fill missing values
X = X.fillna(0)

# Ensure all feature columns are float32 before splitting
X = X.astype(np.float32)

print("Final feature shape:", X.shape)
print("X dtypes before split:")
print(X.dtypes)

# Encode labels (High/Medium/Low -> 0/1/2)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Stratified split because dataset is small
X_train, X_valid, y_train, y_valid = train_test_split(
    X.values,  # X is already float32, so .values will be float32
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)

print("Train shape:", X_train.shape, "Valid shape:", X_valid.shape)
print("Classes:", label_encoder.classes_)


Final feature shape: (255, 34)
X dtypes before split:
Year                          float32
Dacoity                       float32
Robbery                       float32
Murder                        float32
Speedy Trial                  float32
Riot                          float32
Woman & Child Repression      float32
Kidnapping                    float32
Police Assault                float32
Burglary                      float32
Theft                         float32
Other Cases                   float32
Recovery_Case_Arms_Act        float32
Recovery_Case_Explosive       float32
Recovery_Case_Narcotics       float32
Recovery_Case_Smuggling       float32
Crime_Score                   float32
Recovery_Score                float32
Unit Name_Barisal Range       float32
Unit Name_CMP                 float32
Unit Name_Chittagong Range    float32
Unit Name_DMP                 float32
Unit Name_Dhaka Range         float32
Unit Name_GMP                 float32
Unit Name_KMP                 floa

In [19]:
# =============================
# 2. TabNet for BOTH Crime & Recovery (no train/test split)
# =============================

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

# -----------------------------
# Encode target variables
# -----------------------------
le_crime = LabelEncoder()
le_recovery = LabelEncoder()

df['Crime_Level_Encoded'] = le_crime.fit_transform(df['Crime_Level'])
df['Recovery_Level_Encoded'] = le_recovery.fit_transform(df['Recovery_Level'])

print("Crime Level Classes:", list(le_crime.classes_))
print("Recovery Level Classes:", list(le_recovery.classes_))

# -----------------------------
# Select feature columns
# (similar idea to your KNN code)
# -----------------------------
exclude_columns = [
    'Unit Name', 'Year',
    'Crime_Level', 'Recovery_Level',
    'Crime_Level_Encoded', 'Recovery_Level_Encoded'
]

feature_columns = [col for col in df.columns if col not in exclude_columns]

print(f"\nNumber of feature columns: {len(feature_columns)}")
print("Feature columns:", feature_columns)

X_full = df[feature_columns].values.astype(np.float32)

# -----------------------------
# Helper: train + evaluate TabNet
# on full data (train = test)
# -----------------------------
def train_tabnet_for_target(X, y, label_encoder, task_name=""):
    print("\n" + "="*60)
    print(f"TABNET FOR {task_name.upper()} PREDICTION")
    print("="*60)

    clf = TabNetClassifier(
        n_d=32,
        n_a=32,
        n_steps=5,
        gamma=1.5,
        n_independent=2,
        n_shared=2,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=1e-3),
        verbose=0,
        seed=42
    )

    # ⚠️ No train/valid split – training and evaluation on same data
    clf.fit(
        X, y,
        eval_set=[(X, y)],
        eval_name=["train"],
        eval_metric=["accuracy"],
        max_epochs=500,
        patience=50,
        batch_size=64,
        virtual_batch_size=32,
        num_workers=0,
        drop_last=False
    )

    # Predictions on the same data
    y_pred = clf.predict(X)

    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y, y_pred, average='weighted', zero_division=0)

    print(f"\n--- {task_name} METRICS (train = test, may overfit) ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nDetailed classification report:")
    print(classification_report(
        y,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    # Feature importance
    print("\nTop 10 Most Important Features for", task_name)
    importances = clf.feature_importances_
    for name, imp in sorted(zip(feature_columns, importances),
                            key=lambda x: -x[1])[:10]:
        print(f"{name:30s} {imp:.4f}")

    return clf, (acc, prec, rec, f1)

# -----------------------------
# TabNet for Crime Level
# -----------------------------
y_crime = df['Crime_Level_Encoded'].values
tabnet_crime, metrics_crime = train_tabnet_for_target(
    X_full, y_crime, le_crime, task_name="Crime Level"
)

# -----------------------------
# TabNet for Recovery Level
# -----------------------------
y_recovery = df['Recovery_Level_Encoded'].values
tabnet_recovery, metrics_recovery = train_tabnet_for_target(
    X_full, y_recovery, le_recovery, task_name="Recovery Level"
)

# -----------------------------
# Summary table for both tasks
# -----------------------------
print("\n" + "="*60)
print("TABNET MODEL COMPARISON SUMMARY")
print("="*60)

summary_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Crime_Level': metrics_crime,
    'Recovery_Level': metrics_recovery
})

print(summary_df.round(4))


Crime Level Classes: ['High', 'Low', 'Medium']
Recovery Level Classes: ['High', 'Low', 'Medium']

Number of feature columns: 17
Feature columns: ['Dacoity', 'Robbery', 'Murder', 'Speedy Trial', 'Riot', 'Woman & Child Repression', 'Kidnapping', 'Police Assault', 'Burglary', 'Theft', 'Other Cases', 'Recovery_Case_Arms_Act', 'Recovery_Case_Explosive', 'Recovery_Case_Narcotics', 'Recovery_Case_Smuggling', 'Crime_Score', 'Recovery_Score']

TABNET FOR CRIME LEVEL PREDICTION

Early stopping occurred at epoch 199 with best_epoch = 149 and best_train_accuracy = 0.93725

--- Crime Level METRICS (train = test, may overfit) ---
Accuracy : 0.9373
Precision: 0.9380
Recall   : 0.9373
F1-score : 0.9375

Detailed classification report:
              precision    recall  f1-score   support

        High       1.00      0.98      0.99        87
         Low       0.91      0.91      0.91        76
      Medium       0.90      0.92      0.91        92

    accuracy                           0.94       255

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 300 with best_epoch = 250 and best_train_accuracy = 0.96078

--- Recovery Level METRICS (train = test, may overfit) ---
Accuracy : 0.9608
Precision: 0.9627
Recall   : 0.9608
F1-score : 0.9609

Detailed classification report:
              precision    recall  f1-score   support

        High       0.99      0.95      0.97        88
         Low       0.99      0.93      0.96        76
      Medium       0.92      0.99      0.95        91

    accuracy                           0.96       255
   macro avg       0.96      0.96      0.96       255
weighted avg       0.96      0.96      0.96       255


Top 10 Most Important Features for Recovery Level
Recovery_Score                 0.1561
Recovery_Case_Narcotics        0.1150
Speedy Trial                   0.0789
Theft                          0.0768
Murder                         0.0720
Police Assault                 0.0713
Riot                           0.0529
Woman & Child Repression       0.0502
Other

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [20]:
# =============================
# 2. TabNet for BOTH Crime & Recovery (with train/test split)
# =============================
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from sklearn.model_selection import train_test_split

# -----------------------------
# Encode target variables
# -----------------------------
le_crime = LabelEncoder()
le_recovery = LabelEncoder()

df['Crime_Level_Encoded'] = le_crime.fit_transform(df['Crime_Level'])
df['Recovery_Level_Encoded'] = le_recovery.fit_transform(df['Recovery_Level'])

print("Crime Level Classes:", list(le_crime.classes_))
print("Recovery Level Classes:", list(le_recovery.classes_))

# -----------------------------
# Select feature columns (same idea as your KNN code)
# -----------------------------
exclude_columns = [
    'Unit Name', 'Year',
    'Crime_Level', 'Recovery_Level',
    'Crime_Level_Encoded', 'Recovery_Level_Encoded'
]

feature_columns = [col for col in df.columns if col not in exclude_columns]

print(f"\nNumber of feature columns: {len(feature_columns)}")
print("Feature columns:", feature_columns)

X_full = df[feature_columns].values.astype(np.float32)

# -----------------------------
# Helper: train + evaluate TabNet on train/test split
# -----------------------------
def train_tabnet_for_target(X, y, label_encoder, task_name=""):
    print("\n" + "="*60)
    print(f"TABNET FOR {task_name.upper()} PREDICTION")
    print("="*60)

    # Split into train / test (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

    clf = TabNetClassifier(
        n_d=32,
        n_a=32,
        n_steps=5,
        gamma=1.5,
        n_independent=2,
        n_shared=2,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=1e-3),
        verbose=1,           # <--- show epoch logs
        seed=42
    )

    # Train with train & test as eval_set to see logs for both
    clf.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_test, y_test)],
        eval_name=["train", "valid"],
        eval_metric=["accuracy"],
        max_epochs=500,
        patience=50,
        batch_size=64,
        virtual_batch_size=32,
        num_workers=0,
        drop_last=False
    )

    # Predictions on the TEST set (not train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f"\n--- {task_name} TEST METRICS ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nDetailed classification report (TEST):")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    # Feature importance
    print("\nTop 10 Most Important Features for", task_name)
    importances = clf.feature_importances_
    for name, imp in sorted(zip(feature_columns, importances),
                            key=lambda x: -x[1])[:10]:
        print(f"{name:30s} {imp:.4f}")

    # Return model, metrics, and test data (if you want to use them later)
    return clf, (acc, prec, rec, f1), (X_train, X_test, y_train, y_test)

# -----------------------------
# TabNet for Crime Level
# -----------------------------
y_crime = df['Crime_Level_Encoded'].values
tabnet_crime, metrics_crime, data_crime = train_tabnet_for_target(
    X_full, y_crime, le_crime, task_name="Crime Level"
)

# -----------------------------
# TabNet for Recovery Level
# -----------------------------
y_recovery = df['Recovery_Level_Encoded'].values
tabnet_recovery, metrics_recovery, data_recovery = train_tabnet_for_target(
    X_full, y_recovery, le_recovery, task_name="Recovery Level"
)

# -----------------------------
# Summary table for both tasks
# -----------------------------
print("\n" + "="*60)
print("TABNET MODEL COMPARISON SUMMARY (TEST SET)")
print("="*60)

summary_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Crime_Level': metrics_crime,
    'Recovery_Level': metrics_recovery
})

print(summary_df.round(4))


Crime Level Classes: ['High', 'Low', 'Medium']
Recovery Level Classes: ['High', 'Low', 'Medium']

Number of feature columns: 17
Feature columns: ['Dacoity', 'Robbery', 'Murder', 'Speedy Trial', 'Riot', 'Woman & Child Repression', 'Kidnapping', 'Police Assault', 'Burglary', 'Theft', 'Other Cases', 'Recovery_Case_Arms_Act', 'Recovery_Case_Explosive', 'Recovery_Case_Narcotics', 'Recovery_Case_Smuggling', 'Crime_Score', 'Recovery_Score']

TABNET FOR CRIME LEVEL PREDICTION
Train shape: (178, 17), Test shape: (77, 17)
epoch 0  | loss: 2.39027 | train_accuracy: 0.38764 | valid_accuracy: 0.35065 |  0:00:00s


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 1  | loss: 1.96422 | train_accuracy: 0.38764 | valid_accuracy: 0.2987  |  0:00:00s
epoch 2  | loss: 1.86412 | train_accuracy: 0.32584 | valid_accuracy: 0.24675 |  0:00:00s
epoch 3  | loss: 2.02456 | train_accuracy: 0.32584 | valid_accuracy: 0.2987  |  0:00:00s
epoch 4  | loss: 1.76642 | train_accuracy: 0.3764  | valid_accuracy: 0.45455 |  0:00:00s
epoch 5  | loss: 1.67534 | train_accuracy: 0.38764 | valid_accuracy: 0.41558 |  0:00:01s
epoch 6  | loss: 1.55546 | train_accuracy: 0.41011 | valid_accuracy: 0.4026  |  0:00:01s
epoch 7  | loss: 1.46668 | train_accuracy: 0.42135 | valid_accuracy: 0.45455 |  0:00:01s
epoch 8  | loss: 1.49067 | train_accuracy: 0.41011 | valid_accuracy: 0.42857 |  0:00:01s
epoch 9  | loss: 1.33021 | train_accuracy: 0.44944 | valid_accuracy: 0.50649 |  0:00:01s
epoch 10 | loss: 1.24553 | train_accuracy: 0.46067 | valid_accuracy: 0.42857 |  0:00:01s
epoch 11 | loss: 1.16137 | train_accuracy: 0.45506 | valid_accuracy: 0.45455 |  0:00:02s
epoch 12 | loss: 1.22

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")



--- Crime Level TEST METRICS ---
Accuracy : 0.9091
Precision: 0.9189
Recall   : 0.9091
F1-score : 0.9100

Detailed classification report (TEST):
              precision    recall  f1-score   support

        High       1.00      0.92      0.96        26
         Low       0.95      0.83      0.88        23
      Medium       0.82      0.96      0.89        28

    accuracy                           0.91        77
   macro avg       0.92      0.90      0.91        77
weighted avg       0.92      0.91      0.91        77


Top 10 Most Important Features for Crime Level
Recovery_Case_Arms_Act         0.0916
Recovery_Case_Explosive        0.0818
Recovery_Case_Narcotics        0.0783
Murder                         0.0728
Speedy Trial                   0.0688
Police Assault                 0.0666
Theft                          0.0659
Kidnapping                     0.0639
Riot                           0.0613
Burglary                       0.0610

TABNET FOR RECOVERY LEVEL PREDICTION
Train s

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
